# Bunched-Beam Effective Perveance Analysis

$$K_{\rm eff,peak}/K_0 \approx 1 - \bar{\eta}/B_f$$

Primary interpretation limit for any H2/Kr neutralisation result.


In [ ]:
from pathlib import Path
import os, sys, subprocess, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate project root
_ROOT = Path.cwd()
while _ROOT.name != 'plasma_column' and _ROOT.parent != _ROOT:
    _ROOT = _ROOT.parent
if str(_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_ROOT / 'src'))

WORK           = Path.home() / 'Work' / 'simulation_codes-working'
WARPX_DATA_DIR = WORK / 'warpx-data'
RUNS_DIR       = _ROOT / 'runs'
PLOTS_DIR      = _ROOT / 'plots'
RUNS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

os.environ['WARPX_DATA_DIR']  = str(WARPX_DATA_DIR)
os.environ['LD_LIBRARY_PATH'] = (
    str(WORK / 'warpx' / 'install' / 'lib') + ':'
    + os.environ.get('LD_LIBRARY_PATH', '')
)
print('Python :', sys.executable)
print('ROOT   :', _ROOT)
print('WarpX data:', WARPX_DATA_DIR)


In [ ]:
from plasma_column.notebook_utils import print_simulation_config
_DEFAULTS = {
    'beam energy [keV]':       30.0,
    'beam current [mA]':       10.0,
    'bunching factors B_f':    '1, 2, 3, 5, 8',
    'f_RF [MHz]':              72.0,
    'bunch phase width [deg]': 30.0,
}
print_simulation_config(
    notebook_title='Bunched-Beam Effective Perveance',
    defaults=_DEFAULTS, overrides={},
)


## 1. Beam parameters


In [ ]:
import math
from plasma_column.beam import ProtonBeam
from plasma_column.plotting import setup_publication_style, plot_bunched_beam_keff
setup_publication_style()

beam = ProtonBeam(energy_keV=30.0, current_mA=10.0, radius_m=2e-3)
K0   = beam.perveance_K0
B_f_values = [1.0, 2.0, 3.0, 5.0, 8.0]
f_RF_MHz, phase_width_deg = 72.0, 30.0
T_RF = 1.0 / (f_RF_MHz * 1e6)
dt_b = (phase_width_deg / 360.0) * T_RF
dz_b = beam.velocity * dt_b
print(f'K0={K0:.4e}  beta={beam.beta:.6f}  v={beam.velocity:.4e} m/s')
print(f'T_RF={T_RF*1e9:.3f} ns  dt_b={dt_b*1e9:.3f} ns  dz_b={dz_b*1e3:.2f} mm')


## 2. Load eta_avg(t) from first available run


In [ ]:
import warnings
from plasma_column.diagnostics import load_particle_number_diagnostic, compute_particle_number_metrics

_hist = None
for _cd in sorted(RUNS_DIR.iterdir()):
    for _p in [_cd / 'reducedfiles' / 'ParticleNumber_red.txt',
                _cd / 'neutralization_from_particle_number.csv']:
        if _p.exists():
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                _hist = compute_particle_number_metrics(
                    load_particle_number_diagnostic(_p))
            print(f'Using: {_cd.name} ({len(_hist)} steps)')
            break
    if _hist is not None: break

if _hist is None:
    print('No runs found — using synthetic ramp.')
    _t_ns    = np.linspace(0, 400, 300)
    _eta_avg = np.clip(np.linspace(0, 0.75, 300), 0, 1)
else:
    _t_ns    = _hist['time'].values * 1e9
    _eta_avg = _hist['eta_net'].values.clip(0, 1)


In [ ]:
p, _ = plot_bunched_beam_keff(
    _t_ns, _eta_avg, PLOTS_DIR,
    case_name='bunched_beam_analysis',
    bunching_factors=B_f_values,
)
plt.show()
print('Saved:', p.name)


## 3. Final K_eff,peak table


In [ ]:
eta_f = float(_eta_avg[-1])
rows  = [{'B_f': Bf, 'eta_avg_final': eta_f,
           'K_eff_peak_over_K0': max(0.0, 1.0 - eta_f / Bf)}
          for Bf in B_f_values]
display(pd.DataFrame(rows).set_index('B_f').style.format('{:.4f}'))
